# DATA Hypothesis - VANGUARD A/B TEST
---



**How effective was the redesign?**
Database Client Profiles (df_exde): Demographics like age, gender, and account details of our clients. The database is clean.
Web experiment (web): A detailed trace of client interactions online.

IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as st

1. LOAD THE DATASETS 

In [2]:
data_path = "data/processed"
df_exde = pd.read_csv(f"{data_path}/df_exde.csv")
vanguard_client_summary = pd.read_csv(f"{data_path}/vanguard_client_summary.csv")

In [3]:
vanguard_hypo = pd.merge(df_exde,vanguard_client_summary,on=['client_id', 'variation'],how='outer')
vanguard_hypo

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gender,num_accts,balance,calls_6_mnth,logons_6_mnth,variation,age_group,is_completed,is_global_success,has_backwards_error,avg_time_per_step
0,555,3.0,46.0,29.5,unknown,2.0,25454.66,2.0,6.0,Test,young,1,1,0,39.50
1,647,12.0,151.0,57.5,male,2.0,30525.80,0.0,4.0,Test,adult,1,1,0,94.25
2,934,9.0,109.0,51.0,female,2.0,32522.88,0.0,3.0,Test,adult,0,0,0,47.33
3,1028,12.0,145.0,36.0,male,3.0,103520.22,1.0,4.0,Control,adult,0,0,1,67.25
4,1104,5.0,66.0,48.0,unknown,3.0,154643.94,6.0,9.0,Control,adult,0,0,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50495,9999150,5.0,66.0,30.0,unknown,3.0,97141.71,6.0,9.0,Test,adult,0,0,1,9.00
50496,9999400,7.0,86.0,28.5,unknown,2.0,51787.04,0.0,3.0,Test,young,1,1,0,29.75
50497,9999626,9.0,113.0,35.0,male,2.0,36642.88,6.0,9.0,Test,adult,0,0,0,8.00
50498,9999729,10.0,124.0,31.0,female,3.0,107059.74,6.0,9.0,Test,adult,1,0,1,67.50


## Phase 3 — Hypothesis

## Hypothesis 1

**Completion Rate** Completation rate with repetations
H0: Completion rate is the same in the control group and test group.
H1: Completion rate is different in from the control group.

*Choose significance level*

In [4]:
alpha = 0.05

*Compute Test Statistic*

In [5]:
from statsmodels.stats.proportion import proportions_ztest

summary = vanguard_hypo.groupby('variation')['is_completed'].agg(['sum', 'count'])

count = summary['sum'].values
total = summary['count'].values

stat, p = proportions_ztest(count, total)

print("z-stat:", stat)
print("p-value:", p)

z-stat: -8.8745141890702
p-value: 7.023933247581432e-19


In [6]:
print(summary)
#negative z-value: the first group in count has a lower conversion rate than the second group

             sum  count
variation              
Control    15434  23532
Test       18687  26968


*Determine p_value*

In [7]:
p_value = 7.023933247581432e-19

*Decision Making*

In [8]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: Completion rate is the same in the control group and test group")
else:
    print("Rejected the null hypothesis, H1 accepted: Completion rate is different in from the control group.")

Rejected the null hypothesis, H1 accepted: Completion rate is different in from the control group.


**Result Interpretation**: There is significative difference between the test group and the control group (p<0.05). Because the z-stat is minor -8.8745. The Test group has a higher Global completation rate that the control group. 

## Hypothesis 2 with is global_sucess: 
Global completion rate without repetations and errors. 

**Global Completion Rate** Completation rate without repetations
H0: Global Completion rate is the same in the control group and test group.
H1: Global Completion rate is different in from the control group.

In [9]:
alpha = 0.05

In [10]:
from statsmodels.stats.proportion import proportions_ztest

global_summary = vanguard_hypo.groupby('variation')['is_global_success'].agg(['sum', 'count'])

count = global_summary['sum'].values
total = global_summary['count'].values

stat, p = proportions_ztest(count, total)

print("z-stat:", stat)
print("p-value:", p)

z-stat: -1.0416543728319152
p-value: 0.2975719501424685


In [11]:
print(global_summary)
#negative z-value: the first group in count has a lower conversion rate than the second group

            sum  count
variation             
Control    6994  23532
Test       8130  26968


In [12]:
p_value = 0.2975719501424685

In [13]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: Global Completion rate is the same in the control group and test group.")
else:
    print("Rejected the null hypothesis, H1 accepted: Global Completion rate is different in from the control group.")

Null hypothesis accepted, H0: Global Completion rate is the same in the control group and test group.


## Hypothesis 3

**Completion Rate with a Cost-Effectiveness Threshold**
H0: The Test group does not improve completion rate over Control by at least 5%.
H1: The Test group improves completion rate over Control by more than 5%. 

*Choose significance level*

In [14]:
alpha = 0.05

*Compute Test Statistic*

In [15]:
from scipy.stats import norm

# Data
sum_control = 15434
count_control = 23532

sum_test = 18687
count_test = 26968

# Proportions
p_control = sum_control / count_control
p_test = sum_test / count_test

# Pooled proportion (FIXED)
p_pool = (sum_control + sum_test) / (count_control + count_test)

# Standard error
se = np.sqrt(p_pool * (1 - p_pool) * (1/count_control + 1/count_test))

# Threshold (5% absolute uplift)
delta = 0.05

# Z statistic for uplift test
z = ((p_test - p_control) - delta) / se

# One-sided p-value
p_value = 1 - norm.cdf(z)

# Output
print("Control rate:", p_control)
print("Test rate:", p_test)
print("Observed uplift:", p_test - p_control)
print("Z-stat:", z)
print("P-value:", p_value)

Control rate: 0.6558728539860615
Test rate: 0.6929323642835954
Observed uplift: 0.03705951029753385
Z-stat: -3.0988148131491866
P-value: 0.999028517875349


*Determine p_value*

In [16]:
p_value = 0.999028517875349

In [17]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The Test group does not improve completion rate over Control by at least 5%.")
else:
    print("Rejected the null hypothesis, The Test group improves completion rate over Control by more than 5%.")

Null hypothesis accepted, H0: The Test group does not improve completion rate over Control by at least 5%.


**Result Interpretation**: The z-test shows an improvement of completion rate of the test group of 3.7% over the control group. Therefore the H0 is accepted. 

## Hypothesis 4

**Error Rate according to age**
H₀: The error rate is the same in control and test groups
H₁: The error rate differs between control and test groups

*Choose significance level*

In [18]:
alpha = 0.05

*Data Collection*

In [19]:
from statsmodels.stats.proportion import proportions_ztest

error_summary = vanguard_hypo.groupby('variation')['has_backwards_error'].agg(['sum', 'count'])

count = error_summary['sum'].values
total = error_summary['count'].values

stat, p = proportions_ztest(count, total)

print("z-stat:", stat)
print("p-value:", p)


z-stat: -12.097527226948525
p-value: 1.08840485014236e-33


In [20]:
print (error_summary)

             sum  count
variation              
Control     8545  23532
Test       11213  26968


In [21]:
sum_control = 8545
count_control = 23532
sum_test = 11213
count_test = 26968

p_control = sum_control / count_control
p_test = sum_test / count_test
# Output
print("Control rate:", p_control)
print("Test rate:", p_test)

Control rate: 0.36312255651878295
Test rate: 0.415789083358054


*Compute Test Statistic*

In [22]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The error rate is the same in control and test groups")
else:
    print("Rejected the null hypothesis, then H1 accepted: The error rate differs between control and test groups")

Null hypothesis accepted, H0: The error rate is the same in control and test groups


**Result Interpretation**: Even though the total count of errors in the test control is higher showing a negative z-value. The z-test shows the error rate does not have 
a significative difference in both groups (p-value < 0.05). 

## Hypothesis 5

H0: The mean of time spend in each step is the same in Control and Test group
H1: The mean of time spend in each step is the different in control and test group

*Choose significance level*

In [23]:
alpha = 0.05

*Data Collection*

In [24]:
time_test = vanguard_hypo[
    vanguard_hypo['variation'] == "Test"]["avg_time_per_step"].mean()

print(time_test)

72.66970038564224


In [25]:
time_control = vanguard_hypo[
    vanguard_hypo['variation'] == "Control"]["avg_time_per_step"].mean()
print(time_control)

69.87719105898353


*Compute Test Statistic*

In [26]:
from scipy import stats
import pandas as pd

# 1. Extract the WHOLE columns (distributions), not just a single mean
# We use the raw data columns from your dataframe
raw_test = vanguard_hypo[vanguard_hypo['variation'] == 'Test']['avg_time_per_step']
raw_control = vanguard_hypo[vanguard_hypo['variation'] == 'Control']['avg_time_per_step']

# 2. Clean them using pd.to_numeric and dropna
# This works now because 'raw_test' is a Series, not a single float
time_test = pd.to_numeric(raw_test, errors='coerce').dropna()
time_control = pd.to_numeric(raw_control, errors='coerce').dropna()

# 3. Run the T-test
t_stat, p_value = stats.ttest_ind(time_test, time_control, equal_var=False)

print("t-stat:", t_stat)
print("p-value:", p_value)

t-stat: 3.428259400554455
p-value: 0.0006079538909963685


In [27]:
p_value = 0.0006079538909963685
alpha = 0.05

In [28]:
if p_value > alpha:
    print("Null hypothesis accepted, H0: The mean of time spend in each step is the same in Control and Test group")
else:
    print("Rejected the null hypothesis, then H1 accepted: The mean of time spend in each step is the different in control and test group")

Rejected the null hypothesis, then H1 accepted: The mean of time spend in each step is the different in control and test group


**Interpretation results**: The average time spend in each step in significatly different (p<0.05) between groups.

Mann-Whitney test to double check the last hypothesis

In [29]:
from scipy.stats import mannwhitneyu

# Run the test on the cleaned distributions from your previous step
u_stat, p_value = mannwhitneyu(time_test, time_control, alternative='two-sided')

print(f"Mann-Whitney U Statistic: {u_stat}")
print(f"P-value: {p_value}")

# Interpretation
if p_value < 0.05:
    print("Verdict: Statistically Significant difference (Reject H0)")
else:
    print("Verdict: No significant difference (Fail to reject H0)")

Mann-Whitney U Statistic: 322132221.0
P-value: 0.003134566837707991
Verdict: Statistically Significant difference (Reject H0)


In [30]:
import os

# Define the output directory
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

vanguard_hypo.to_csv(f"{output_dir}/vanguard_hypo.csv", index=False)
vanguard_hypo.to_csv(f"{output_dir}/vanguard_hypo.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (3,302,143 bytes)
  df_web_3.csv (81,420,206 bytes)
  vanguard_client_summary.csv (1,354,845 bytes)
  vanguard_hypo.csv (3,884,978 bytes)
  vanguard_web_wrangled.csv (35,857,674 bytes)
  web.csv (40,471,437 bytes)
